# Model Evaluation and Return-Risk Insights

Compare the saved classifiers on the same stratified holdout set. F1 is used to rank models, while precision, recall, and ROC-AUC provide complementary business-focused measures. The winning model and its strongest input features are saved for reuse.

## Load the holdout data and fitted models

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

DATA_PATH = Path("../datasets/ecommerce_return_prediction_feature_engineered.csv")
MODEL_PATH = Path("../models/trained_models.joblib")
OUTPUT_DIR = Path("../models")

df = pd.read_csv(DATA_PATH).drop_duplicates()
X = df.drop(columns=["returned", "return_rate_pct", "order_date"])
y = df["returned"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model_artifact = joblib.load(MODEL_PATH)
models = model_artifact["models"]
if X.columns.tolist() != model_artifact["feature_columns"]:
    raise ValueError("Evaluation features do not match the saved training features.")

print(f"Holdout rows: {len(X_test):,}")
print(f"Saved models loaded: {', '.join(models)}")

## Compare models on the holdout set

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

evaluation_results = []
holdout_predictions = {}

for name, fitted_model in models.items():
    predictions = fitted_model.predict(X_test)
    probabilities = fitted_model.predict_proba(X_test)[:, 1]
    holdout_predictions[name] = (predictions, probabilities)
    evaluation_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probabilities),
    })

evaluation_df = pd.DataFrame(evaluation_results).sort_values(
    "F1", ascending=False
).reset_index(drop=True)
evaluation_df.to_csv(OUTPUT_DIR / "evaluation_metrics.csv", index=False)
display(evaluation_df.style.format({
    "Accuracy": "{:.3f}", "Precision": "{:.3f}", "Recall": "{:.3f}",
    "F1": "{:.3f}", "ROC-AUC": "{:.3f}"
}))

In [ ]:
best_model_name = evaluation_df.loc[0, "Model"]
best_model = models[best_model_name]
best_predictions, best_probabilities = holdout_predictions[best_model_name]

joblib.dump(best_model, OUTPUT_DIR / "best_model.joblib")
prediction_results = pd.DataFrame({
    "actual_returned": y_test.to_numpy(),
    "predicted_returned": best_predictions,
    "return_probability": best_probabilities,
}, index=y_test.index).sort_index()
prediction_results.to_csv(OUTPUT_DIR / "best_model_test_predictions.csv", index=False)

print(f"Best model by holdout F1: {best_model_name}")
print(evaluation_df.iloc[0].to_string())

## Winning-model diagnostics

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, best_predictions, display_labels=["Kept", "Returned"],
    cmap="Blues", ax=axes[0], colorbar=False
)
axes[0].set_title(f"{best_model_name}: confusion matrix")
RocCurveDisplay.from_predictions(y_test, best_probabilities, ax=axes[1])
axes[1].set_title("ROC curve")
PrecisionRecallDisplay.from_predictions(y_test, best_probabilities, ax=axes[2])
axes[2].set_title("Precision-recall curve")
plt.tight_layout()
plt.show()

## Explain the strongest model features

In [ ]:
preprocessor = best_model.named_steps["preprocessor"]
estimator = best_model.named_steps["model"]
feature_names = preprocessor.get_feature_names_out()

if hasattr(estimator, "feature_importances_"):
    importance_values = estimator.feature_importances_
    importance_label = "Feature importance"
elif hasattr(estimator, "coef_"):
    importance_values = np.abs(estimator.coef_).ravel()
    importance_label = "Absolute coefficient"
else:
    raise TypeError(f"Feature explanation is not supported for {best_model_name}.")

feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    importance_label: importance_values,
}).sort_values(importance_label, ascending=False).head(20)
feature_importance_df.to_csv(OUTPUT_DIR / "best_model_feature_importance.csv", index=False)
display(feature_importance_df)

feature_importance_df.sort_values(importance_label).plot.barh(
    x="Feature", y=importance_label, figsize=(9, 7), legend=False,
    title=f"Top return-prediction signals: {best_model_name}"
)
plt.tight_layout()
plt.show()

## Two-model accuracy comparison: Random Forest vs XGBoost

This additional section trains only two tree-based candidates and leaves the evaluation cells above unchanged. The dataset is imbalanced, so an accuracy near the majority-class baseline can hide a model that misses almost every return. This comparison therefore reports accuracy, precision, recall, F1, and ROC-AUC at the default threshold, plus results at a threshold selected on a separate validation split to maximize F1. The accuracy winner is selected by validation accuracy, not by test-set results.

In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
import joblib

two_model_df = pd.read_csv(DATA_PATH).drop_duplicates()
two_model_X = two_model_df.drop(
    columns=["returned", "return_rate_pct", "order_date"]
)
two_model_y = two_model_df["returned"]
two_model_X_train, two_model_X_test, two_model_y_train, two_model_y_test = train_test_split(
    two_model_X,
    two_model_y,
    test_size=0.2,
    random_state=42,
    stratify=two_model_y,
)
two_model_X_fit, two_model_X_val, two_model_y_fit, two_model_y_val = train_test_split(
    two_model_X_train,
    two_model_y_train,
    test_size=0.2,
    random_state=43,
    stratify=two_model_y_train,
)

two_model_numeric = two_model_X.select_dtypes(include="number").columns.tolist()
two_model_categorical = two_model_X.select_dtypes(
    include=["object", "category", "str"]
).columns.tolist()
two_model_preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), two_model_numeric),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]), two_model_categorical),
])

two_model_estimators = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=16,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=2,
        eval_metric="logloss",
        random_state=42,
        n_jobs=1,
        tree_method="hist",
    ),
}

two_model_validation = {}
two_model_fitted = {}
two_model_rows = []
for two_model_name, two_model_estimator in two_model_estimators.items():
    two_model_validation_pipeline = Pipeline([
        ("preprocessor", clone(two_model_preprocessor)),
        ("model", clone(two_model_estimator)),
    ])
    two_model_validation_pipeline.fit(two_model_X_fit, two_model_y_fit)
    two_model_val_prob = two_model_validation_pipeline.predict_proba(
        two_model_X_val
    )[:, 1]
    two_model_val_default = (two_model_val_prob >= 0.5).astype(int)
    two_model_precision, two_model_recall, two_model_thresholds = precision_recall_curve(
        two_model_y_val, two_model_val_prob
    )
    two_model_val_f1 = (
        2 * two_model_precision[:-1] * two_model_recall[:-1]
        / np.maximum(two_model_precision[:-1] + two_model_recall[:-1], 1e-12)
    )
    two_model_best_threshold_index = int(np.nanargmax(two_model_val_f1))
    two_model_f1_threshold = float(
        two_model_thresholds[two_model_best_threshold_index]
    )

    two_model_final_pipeline = Pipeline([
        ("preprocessor", clone(two_model_preprocessor)),
        ("model", clone(two_model_estimator)),
    ])
    two_model_final_pipeline.fit(two_model_X_train, two_model_y_train)
    two_model_test_prob = two_model_final_pipeline.predict_proba(
        two_model_X_test
    )[:, 1]
    two_model_test_default = (two_model_test_prob >= 0.5).astype(int)
    two_model_test_f1_threshold = (
        two_model_test_prob >= two_model_f1_threshold
    ).astype(int)
    two_model_fitted[two_model_name] = two_model_final_pipeline
    two_model_validation[two_model_name] = {
        "validation_accuracy": accuracy_score(two_model_y_val, two_model_val_default),
        "validation_f1": f1_score(
            two_model_y_val,
            (two_model_val_prob >= two_model_f1_threshold).astype(int),
            zero_division=0,
        ),
        "f1_threshold": two_model_f1_threshold,
    }
    two_model_rows.append({
        "Model": two_model_name,
        "Validation Accuracy (0.50)": accuracy_score(
            two_model_y_val, two_model_val_default
        ),
        "F1 Threshold (validation)": two_model_f1_threshold,
        "Test Accuracy (0.50)": accuracy_score(
            two_model_y_test, two_model_test_default
        ),
        "Test Precision (0.50)": precision_score(
            two_model_y_test, two_model_test_default, zero_division=0
        ),
        "Test Recall (0.50)": recall_score(
            two_model_y_test, two_model_test_default, zero_division=0
        ),
        "Test F1 (0.50)": f1_score(
            two_model_y_test, two_model_test_default, zero_division=0
        ),
        "Test Accuracy (F1 threshold)": accuracy_score(
            two_model_y_test, two_model_test_f1_threshold
        ),
        "Test Precision (F1 threshold)": precision_score(
            two_model_y_test, two_model_test_f1_threshold, zero_division=0
        ),
        "Test Recall (F1 threshold)": recall_score(
            two_model_y_test, two_model_test_f1_threshold, zero_division=0
        ),
        "Test F1 (F1 threshold)": f1_score(
            two_model_y_test, two_model_test_f1_threshold, zero_division=0
        ),
        "Test ROC-AUC": roc_auc_score(two_model_y_test, two_model_test_prob),
    })

two_model_metrics = pd.DataFrame(two_model_rows).sort_values(
    "Validation Accuracy (0.50)", ascending=False
).reset_index(drop=True)
two_model_metrics.to_csv(OUTPUT_DIR / "two_model_comparison_metrics.csv", index=False)
two_model_artifact = {
    "models": two_model_fitted,
    "feature_columns": two_model_X.columns.tolist(),
    "validation_thresholds": {
        name: values["f1_threshold"]
        for name, values in two_model_validation.items()
    },
    "validation_f1": {
        name: values["validation_f1"]
        for name, values in two_model_validation.items()
    },
}
joblib.dump(two_model_artifact, OUTPUT_DIR / "two_model_comparison.joblib")

two_model_accuracy_winner = two_model_metrics.loc[0, "Model"]
joblib.dump(
    two_model_fitted[two_model_accuracy_winner],
    OUTPUT_DIR / "best_accuracy_model.joblib",
)
display(two_model_metrics.style.format({
    column: "{:.3f}"
    for column in two_model_metrics.columns
    if column != "Model"
}))

two_model_majority_baseline = max(
    two_model_y_test.mean(), 1 - two_model_y_test.mean()
)
print(f"Test majority-class accuracy baseline: {two_model_majority_baseline:.3f}")
print(f"Best model by validation accuracy: {two_model_accuracy_winner}")
print("The F1-threshold columns show the accuracy/return-recall trade-off.")